In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Go to Runtime > Change runtime type > T4 GPU, then re-run."
print("GPU:", torch.cuda.get_device_name(0))

!pip -q install -U bitsandbytes accelerate

from google.colab import drive
drive.mount('/content/drive')

import os, glob

def find_project_dir(name="dl-quant_project"):
    candidates = [
        f"/content/drive/MyDrive/Colab Notebooks/{name}",
        f"/content/drive/MyDrive/{name}",
        *glob.glob(f"/content/drive/MyDrive/*/{name}"),
        *glob.glob(f"/content/drive/Shareddrives/*/{name}"),
    ]
    for p in candidates:
        if os.path.isdir(p):
            return p
    raise FileNotFoundError("Could not find dl-quant_project. Add a shortcut to it in My Drive.")

PROJECT_DIR = find_project_dir()
DATA_DIR = os.path.join(PROJECT_DIR, "DataSource")
print("Data folder:", DATA_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 110)
SEED = 42

GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data folder: /content/drive/MyDrive/Colab Notebooks/dl-quant_project/DataSource


Step 1: Load the data

In [ ]:
TEXT_MODE = "headline_lead"          # "headline" or "headline_lead"
LEAD_CHARS = 300                # used only for "headline_lead"

news = pd.read_csv(os.path.join(DATA_DIR, "fnspid_meme_clean.csv"),
                   parse_dates=["date", "target_date"])
news["headline"] = news["Article_title"].fillna("").str.strip()
news = news[news["headline"].str.len() > 0]

# The "text" column from notebook 01 is: headline + ". " + first 1,500 characters of the body
def get_lead(row):
    body = str(row["text"])[len(row["headline"]) + 2:].strip()
    return body[:LEAD_CHARS]

if TEXT_MODE == "headline_lead":
    news["lead"] = news.apply(get_lead, axis=1)
    news["input_text"] = np.where(news["lead"].str.len() > 0,
                                  news["headline"] + " — " + news["lead"],
                                  news["headline"])
else:
    news["input_text"] = news["headline"]

before = len(news)
news = (news.sort_values("date")
            .drop_duplicates(subset=["ticker", "headline"], keep="first")
            .reset_index(drop=True))
print(f"Headlines: {before} -> {len(news)} after removing repeats")
print(news.groupby("ticker").size())
print("\nExample input texts:")
print(news.sample(3, random_state=SEED)["input_text"].to_string())

!pip -q install datasets
from datasets import load_dataset
tw = load_dataset("zeroshot/twitter-financial-news-sentiment", split="validation").to_pandas()
ID2LABEL = {0: "bearish", 1: "bullish", 2: "neutral"}
tw["true"] = tw["label"].map(ID2LABEL)
print("\nTwitter validation:", len(tw), "texts")
print(tw["true"].value_counts(normalize=True).round(3))
print(tw.sample(3, random_state=SEED)[["text", "true"]].to_string())

Headlines: 9538 -> 8784 after removing repeats
ticker
AMC    3027
BB     1071
GME    4686
dtype: int64

Example input texts:
6385    Too Late to Join BB Stock Rally as Upside Is Gone — InvestorPlace - Stock Market News, Stock Advice & Trad...
1488    Notable Tuesday Option Activity: GME, SSYS, BURL — Among the underlying components of the Russell 3000 ind...
6539    Meme Stocks Rally: What Reddit Investors Are Saying About AMC, GME, EXPR, KOSS and BB Today — InvestorPlac...

Twitter validation: 2388 texts
true
neutral    0.656
bullish    0.199
bearish    0.145
Name: proportion, dtype: float64
                                                                                                                                              text     true
544   Two key regulators are holding off on easing Wall Street debt limits, leaving billions of dollars locked up that co… https://t.co/YvDDlEcmh2  neutral
1911              Xinjiang Xinxin Mining Industry : ANNOUNCEMENT - RESIGNATION OF GENERA

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"   # faster alternative: "Qwen/Qwen2.5-3B-Instruct"
PROMPT_VERSION = "v1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True,
                         bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME,
                                             quantization_config=bnb,
                                             device_map="auto",
                                             low_cpu_mem_usage=True)
model.eval()
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

ModuleNotFoundError: Could not import module 'Qwen2ForCausalLM'. Are this object's requirements defined correctly? Set the logging verbosity to DEBUG for the original import error.

Step 3: Turn the LLM into a classifier

In [ ]:
COMPANY = {"GME": "GameStop", "AMC": "AMC Entertainment", "BB": "BlackBerry"}
LABELS = ["bearish", "neutral", "bullish"]

SYSTEM = ("You are a careful financial analyst. You classify the sentiment of financial news "
          "for investors in a specific stock.")

def build_prompt(text, ticker=None):
    stock = f"{ticker} ({COMPANY.get(ticker, ticker)})" if ticker else "the stock or market mentioned"
    holder = ticker if ticker else "that stock"
    user = (f"Stock: {stock}\n"
            f"News: {text}\n\n"
            f"From the point of view of an investor holding {holder}, is this news bearish, neutral, or bullish?\n"
            "- bullish: good news, likely positive for the price\n"
            "- bearish: bad news, likely negative for the price\n"
            "- neutral: factual, mixed, or not clearly about the stock's prospects\n"
            "Answer with exactly one word: bearish, neutral, or bullish.")
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": user}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print(build_prompt("GameStop shares soar 50% as Reddit traders pile in", "GME"))

In [ ]:
def first_token_ids(word):
    ids = set()
    for variant in [word, word.capitalize(), " " + word, " " + word.capitalize()]:
        ids.add(tokenizer.encode(variant, add_special_tokens=False)[0])
    return sorted(ids)

LABEL_TOKEN_IDS = {lab: first_token_ids(lab) for lab in LABELS}
print({lab: [tokenizer.decode([i]) for i in ids] for lab, ids in LABEL_TOKEN_IDS.items()})

all_ids = [i for ids in LABEL_TOKEN_IDS.values() for i in ids]
assert len(all_ids) == len(set(all_ids)), "Answer tokens overlap between labels"

In [ ]:
COMPANY = {"GME": "GameStop", "AMC": "AMC Entertainment", "BB": "BlackBerry"}
LABELS = ["bearish", "neutral", "bullish"]

SYSTEM = ("You are a careful financial analyst. You classify the sentiment of financial news "
          "for investors in a specific stock.")

def build_prompt(text, ticker=None):
    stock = f"{ticker} ({COMPANY.get(ticker, ticker)})" if ticker else "the stock or market mentioned"
    holder = ticker if ticker else "that stock"
    user = (f"Stock: {stock}\n"
            f"News: {text}\n\n"
            f"From the point of view of an investor holding {holder}, is this news bearish, neutral, or bullish?\n"
            "- bullish: good news, likely positive for the price\n"
            "- bearish: bad news, likely negative for the price\n"
            "- neutral: factual, mixed, or not clearly about the stock's prospects\n"
            "Answer with exactly one word: bearish, neutral, or bullish.")
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": user}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print(build_prompt("GameStop shares soar 50% as Reddit traders pile in", "GME"))

In [ ]:
@torch.no_grad()
def llm_probs(texts, tickers=None, batch_size=16):
    tickers = tickers if tickers is not None else [None] * len(texts)
    out = []
    for i in range(0, len(texts), batch_size):
        prompts = [build_prompt(x, t) for x, t in
                   zip(texts[i:i + batch_size], tickers[i:i + batch_size])]
        enc = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)

        logits = model(**enc).logits[:, -1, :].float()
        logprobs = torch.log_softmax(logits, dim=-1)

        per_label = torch.stack([torch.logsumexp(logprobs[:, ids], dim=-1)
                                 for ids in LABEL_TOKEN_IDS.values()], dim=-1)
        out.append(torch.softmax(per_label, dim=-1).cpu().numpy())
    return np.vstack(out)    # columns: p_bear, p_neu, p_bull

demo = news.sample(6, random_state=SEED)
p = llm_probs(demo["input_text"].tolist(), demo["ticker"].tolist())
print(pd.DataFrame({"ticker": demo["ticker"].values,
                    "text": demo["input_text"].str[:90].values,
                    "label": np.array(LABELS)[p.argmax(1)],
                    "p_bear": p[:, 0].round(2), "p_neu": p[:, 1].round(2), "p_bull": p[:, 2].round(2)})
      .to_string(index=False))

Step 4: check whether we can believe the llm

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import time

t0 = time.time()
p_tw = llm_probs(tw["text"].tolist())          # no ticker: Twitter texts cover many stocks
tw["llm"] = np.array(LABELS)[p_tw.argmax(1)]
tw["llm_conf"] = p_tw.max(1)
print(f"Labeled {len(tw)} texts in {time.time() - t0:.0f} s")

acc = accuracy_score(tw["true"], tw["llm"])
mf1 = f1_score(tw["true"], tw["llm"], average="macro")
print(f"\nLLM accuracy: {acc:.3f}   macro-F1: {mf1:.3f}")
print("Baselines:    always-neutral 0.656 accuracy | keyword 0.621 accuracy, 0.504 macro-F1\n")

print(classification_report(tw["true"], tw["llm"], labels=LABELS, digits=3))
print(pd.DataFrame(confusion_matrix(tw["true"], tw["llm"], labels=LABELS),
                   index=[f"true_{l}" for l in LABELS],
                   columns=[f"pred_{l}" for l in LABELS]))

tw["conf_bin"] = pd.cut(tw["llm_conf"], [0, 0.5, 0.7, 0.9, 1.0])
print("\nAccuracy by LLM confidence:")
print(tw.groupby("conf_bin", observed=True)
        .apply(lambda d: pd.Series({"n": len(d), "accuracy": (d["true"] == d["llm"]).mean()}))
        .round(3))

Step 4b Calibration

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, log_loss

# ---- 1. Split the Twitter data into two halves (same label mix in each) ----
idx_a, idx_b = train_test_split(np.arange(len(tw)), test_size=0.5,
                                stratify=tw["true"], random_state=SEED)
P = np.clip(p_tw, 1e-12, 1.0)                    # avoid log(0)
y = tw["true"].map({lab: i for i, lab in enumerate(LABELS)}).to_numpy()

def calibrate(P, T=1.0, w_neutral=1.0):
    # Temperature T softens (T > 1) the probabilities; w_neutral boosts neutral
    logits = np.log(P) / T + np.log(np.array([1.0, w_neutral, 1.0]))
    Q = np.exp(logits - logits.max(axis=1, keepdims=True))
    return Q / Q.sum(axis=1, keepdims=True)

# ---- 2. Temperature: pick T on half A by minimizing log loss ----
temps = [1, 1.5, 2, 3, 4, 6, 8, 10, 15, 20]
loss_by_T = {T: log_loss(y[idx_a], calibrate(P[idx_a], T=T), labels=[0, 1, 2]) for T in temps}
T_best = min(loss_by_T, key=loss_by_T.get)
print("Log loss on half A by temperature:", {T: round(v, 3) for T, v in loss_by_T.items()})
print("Chosen temperature T =", T_best)

# ---- 3. Neutral weight: pick w on half A by maximizing macro-F1 ----
weights = [1, 1.5, 2, 3, 4, 6, 8, 10, 15, 20, 30]
f1_by_w = {w: f1_score(y[idx_a], calibrate(P[idx_a], T_best, w).argmax(1), average="macro")
           for w in weights}
w_best = max(f1_by_w, key=f1_by_w.get)
print("\nMacro-F1 on half A by neutral weight:", {w: round(v, 3) for w, v in f1_by_w.items()})
print("Chosen neutral weight w =", w_best)

# ---- 4. Honest check on half B (not used for choosing T or w) ----
def report(Q, name):
    pred = Q.argmax(1)
    conf = Q.max(1)
    bins = pd.cut(conf, [0, 0.5, 0.7, 0.9, 1.0])
    rel = (pd.DataFrame({"bin": bins, "conf": conf, "correct": pred == y[idx_b]})
             .groupby("bin", observed=True)
             .agg(n=("correct", "size"), mean_confidence=("conf", "mean"), accuracy=("correct", "mean"))
             .round(3))
    print(f"\n=== {name} ===")
    print(f"accuracy {accuracy_score(y[idx_b], pred):.3f} | macro-F1 {f1_score(y[idx_b], pred, average='macro'):.3f} "
          f"| log loss {log_loss(y[idx_b], Q, labels=[0, 1, 2]):.3f}")
    print("Predicted label mix:", pd.Series(np.array(LABELS)[pred]).value_counts(normalize=True).round(2).to_dict())
    print("True label mix:     ", tw["true"].iloc[idx_b].value_counts(normalize=True).round(2).to_dict())
    print(rel)

report(calibrate(P[idx_b]), "Half B, before calibration")
report(calibrate(P[idx_b], T_best, w_best), f"Half B, after calibration (T={T_best}, w={w_best})")

Step 5a Check the lads for boilerplate

In [ ]:
T_CAL = 4      # temperature from Step 4b
W_CAL = 4      # neutral weight (4 suggested; Step 4b picked 6)

has_lead = news["lead"].str.len() > 0
print(f"Headlines with a lead: {has_lead.mean():.1%}")

lead_starts = news.loc[has_lead, "lead"].str[:60]
print("\nMost common lead beginnings:")
print(lead_starts.value_counts().head(20).to_string())

Step 5b. Clean the leads

In [ ]:
BOILERPLATE = [
    r"^(InvestorPlace\s*)+-\s*Stock Market News,\s*Stock Advice\s*&\s*Trading Tips\s*",
    r"^Eikon search string for individual stock moves:\s*STXBZ\s*",
    r"^The Day Ahead newsletter:\s*",
    r"^The Morning News Call newsletter:\s*",
    r"^What happened\s*",
    r"^\(RTTNews\)\s*-\s*",
]

def clean_body(body):
    body = re.sub(r"https?://\S+", " ", str(body))       # NEW: remove web links
    body = re.sub(r"\s+", " ", body).strip()
    changed = True
    while changed:
        changed = False
        for pat in BOILERPLATE:
            new = re.sub(pat, "", body, flags=re.IGNORECASE).strip()
            if new != body:
                body, changed = new, True
    return body

def get_clean_lead(row):
    body = str(row["text"])[len(row["headline"]) + 2:]   # skip "headline. "
    return clean_body(body)[:LEAD_CHARS]

news["lead"] = news.apply(get_clean_lead, axis=1)
news["input_text"] = np.where(news["lead"].str.len() > 0,
                              news["headline"] + " — " + news["lead"],
                              news["headline"])

print("Most common lead beginnings after cleaning:")
print(news.loc[news["lead"].str.len() > 0, "lead"].str[:60].value_counts().head(10).to_string())

print("\nExample (an article that had InvestorPlace boilerplate):")
ex = news[news["text"].str.contains("InvestorPlace - Stock Market News", na=False)].iloc[0]
print(ex["input_text"])

Step 5c. Label all headlines (with checkpoints)

In [ ]:
PARTIAL = os.path.join(DATA_DIR, f"llm_labels_partial_{TEXT_MODE}_{PROMPT_VERSION}.csv")
CHUNK = 500

news["key"] = news["ticker"] + "||" + news["headline"]

if os.path.exists(PARTIAL):
    done = pd.read_csv(PARTIAL)
else:
    done = pd.DataFrame(columns=["key", "p_bear_raw", "p_neu_raw", "p_bull_raw"])
done = done[done["key"].isin(news["key"])]
todo = news[~news["key"].isin(done["key"])]
print(f"Already labeled: {len(done)} | to do: {len(todo)}")
print(f"Saving progress to: {PARTIAL}\n")

t0 = time.time()
for start in range(0, len(todo), CHUNK):
    part = todo.iloc[start:start + CHUNK]
    p = llm_probs(part["input_text"].tolist(), part["ticker"].tolist())
    new = pd.DataFrame({"key": part["key"].values,
                        "p_bear_raw": p[:, 0], "p_neu_raw": p[:, 1], "p_bull_raw": p[:, 2]})
    done = pd.concat([done, new], ignore_index=True)
    done.to_csv(PARTIAL, index=False)                     # checkpoint after every chunk

    finished = start + len(part)
    rate = finished / (time.time() - t0)
    minutes_left = (len(todo) - finished) / rate / 60
    print(f"{len(done):5d}/{len(news)} labeled | {rate:.1f} headlines/s | ~{minutes_left:.0f} min left")

print("\nDone. Total time: %.0f min" % ((time.time() - t0) / 60))

Step 6a Apply calibration and inspect the labels

In [ ]:
def calibrate(P, T=1.0, w_neutral=1.0):          # same function as Step 4b (redefined in case of a restart)
    logits = np.log(np.clip(P, 1e-12, 1.0)) / T + np.log(np.array([1.0, w_neutral, 1.0]))
    Q = np.exp(logits - logits.max(axis=1, keepdims=True))
    return Q / Q.sum(axis=1, keepdims=True)

labels = news.merge(done, on="key", how="inner")
P_raw = labels[["p_bear_raw", "p_neu_raw", "p_bull_raw"]].to_numpy()
P_cal = calibrate(P_raw, T_CAL, W_CAL)

labels["llm_label_raw"] = np.array(LABELS)[P_raw.argmax(1)]
labels[["p_bear", "p_neu", "p_bull"]] = P_cal
labels["llm_label"] = np.array(LABELS)[P_cal.argmax(1)]
labels["llm_confidence"] = P_cal.max(1)
print(f"Labeled headlines: {len(labels)}  (calibration: T={T_CAL}, w={W_CAL})")

# 1. Raw vs calibrated label mix
print("\nLabel mix, raw vs calibrated:")
print(pd.DataFrame({"raw": labels["llm_label_raw"].value_counts(normalize=True),
                    "calibrated": labels["llm_label"].value_counts(normalize=True)}).round(3))
print(f"Labels changed by calibration: {(labels['llm_label'] != labels['llm_label_raw']).mean():.1%}")
print(pd.crosstab(labels["llm_label_raw"], labels["llm_label"],
                  rownames=["raw"], colnames=["calibrated"]))

# 2. By stock and by year
print("\nCalibrated label mix by stock:")
print(pd.crosstab(labels["ticker"], labels["llm_label"], normalize="index").round(3))
print("\nCalibrated label mix by year:")
print(pd.crosstab(labels["date"].dt.year, labels["llm_label"], normalize="index").round(2))

# 3. Confidence
plt.figure(figsize=(6, 3))
plt.hist(labels["llm_confidence"], bins=30, color="#457b9d")
plt.title("Calibrated LLM confidence on FNSPID headlines"); plt.xlabel("max probability")
plt.tight_layout(); plt.show()

# 4. Examples to read
cols = ["ticker", "date", "headline", "llm_label", "llm_confidence"]
print("\nMost confident bullish:")
print(labels[labels.llm_label == "bullish"].nlargest(5, "llm_confidence")[cols].to_string(index=False))
print("\nMost confident bearish:")
print(labels[labels.llm_label == "bearish"].nlargest(5, "llm_confidence")[cols].to_string(index=False))
print("\nLeast confident (hard cases):")
print(labels.nsmallest(5, "llm_confidence")[cols].to_string(index=False))
print("\nChanged by calibration (random 5):")
changed = labels[labels["llm_label"] != labels["llm_label_raw"]]
print(changed.sample(min(5, len(changed)), random_state=SEED)[cols + ["llm_label_raw"]].to_string(index=False))

In [ ]:
## Make an apple to apple (refer to the above Step 4 for uncalibrated result) comparison
t0 = time.time()
p_tw_raw = llm_probs(tw["text"].tolist())          # raw LLM probabilities (no ticker)
p_tw_cal = calibrate(p_tw_raw, T_CAL, W_CAL)       # calibrated probabilities
print(f"Labeled {len(tw)} texts in {time.time() - t0:.0f} s")

tw["llm_raw"]       = np.array(LABELS)[p_tw_raw.argmax(1)]
tw["llm"]           = np.array(LABELS)[p_tw_cal.argmax(1)]     # calibrated label
tw["llm_conf"]      = p_tw_cal.max(1)
tw[["p_bear_raw", "p_neu_raw", "p_bull_raw"]] = p_tw_raw       # keep for later reuse
tw[["p_bear", "p_neu", "p_bull"]]             = p_tw_cal

print("Baselines:    always-neutral 0.656 accuracy | keyword 0.621 accuracy, 0.504 macro-F1")
print(f"Calibration:  T={T_CAL}, w_neutral={W_CAL}")
print(f"Labels changed by calibration: {(tw['llm'] != tw['llm_raw']).mean():.1%}")

for name, col in [("RAW", "llm_raw"), ("CALIBRATED", "llm")]:
    acc = accuracy_score(tw["true"], tw[col])
    mf1 = f1_score(tw["true"], tw[col], labels=LABELS, average="macro")
    bal = balanced_accuracy_score(tw["true"], tw[col])
    print(f"\n===== {name} =====")
    print(f"accuracy {acc:.3f}   macro-F1 {mf1:.3f}   balanced acc {bal:.3f}")
    print(classification_report(tw["true"], tw[col], labels=LABELS, digits=3))
    print(pd.DataFrame(confusion_matrix(tw["true"], tw[col], labels=LABELS),
                       index=[f"true_{l}" for l in LABELS],
                       columns=[f"pred_{l}" for l in LABELS]))

In [ ]:
from scipy import stats

# 1. A closer look at a suspicious label
row = labels[labels["headline"].str.contains("Reports Q4 Loss", na=False)].iloc[0]
print("Input text the LLM saw:\n", row["input_text"])
print("\nCalibrated probabilities:", row[["p_bear", "p_neu", "p_bull"]].round(2).to_dict())

# 2. Daily returns per stock
px = pd.read_csv(os.path.join(DATA_DIR, "prices_gme_amc_bb.csv"),
                 header=[0, 1], index_col=0, parse_dates=True)
rets = []
for t in COMPANY:
    s = px[("Adj Close", t)].dropna()
    rets.append(pd.DataFrame({"ticker": t,
                              "trade_day": s.index.normalize(),
                              "ret": np.log(s).diff().values}))
rets = pd.concat(rets).dropna()

days = np.array(sorted(rets["trade_day"].unique()), dtype="datetime64[ns]")
pos = np.searchsorted(days, labels["date"].values.astype("datetime64[ns]"), side="left")
ok = pos < len(days)
chk = labels[ok].assign(trade_day=days[pos[ok]]).merge(rets, on=["ticker", "trade_day"])

# Keyword labels from notebook 02, for comparison
kw = pd.read_csv(os.path.join(DATA_DIR, "articles_scored_keyword.csv"), usecols=["ticker", "Url", "label"])
chk = chk.merge(kw.rename(columns={"label": "kw_label"}).drop_duplicates(["ticker", "Url"]),
                on=["ticker", "Url"], how="left")

# 3. Spread for each label set
def spread(df, col):
    bull = df.loc[df[col] == "bullish", "ret"]
    bear = df.loc[df[col] == "bearish", "ret"]
    t, p = stats.ttest_ind(bull, bear, equal_var=False)
    return pd.Series({"n_bull": len(bull), "n_bear": len(bear),
                      "bull_mean_%": 100 * bull.mean(), "bear_mean_%": 100 * bear.mean(),
                      "spread_pp": 100 * (bull.mean() - bear.mean()), "t": t, "p": p})

result = pd.DataFrame({name: spread(chk, col) for name, col in
                       [("LLM raw", "llm_label_raw"), ("LLM calibrated", "llm_label"), ("Keyword", "kw_label")]}).T
print("\nSame-day return by label (all years):")
print(result.round(3))

# 4. By period
chk["period"] = pd.cut(chk["date"].dt.year, [2008, 2020, 2021, 2023], labels=["2009-2020", "2021", "2022-2023"])
rows = []
for per, d in chk.groupby("period", observed=True):
    for name, col in [("LLM raw", "llm_label_raw"), ("LLM calibrated", "llm_label"), ("Keyword", "kw_label")]:
        s = spread(d, col)
        rows.append({"period": per, "labels": name, "spread_pp": s["spread_pp"], "p": s["p"],
                     "n_bull": s["n_bull"], "n_bear": s["n_bear"]})
print("\nSpread by period:")
print(pd.DataFrame(rows).pivot(index="period", columns="labels", values="spread_pp").round(2))
print("\np-values by period:")
print(pd.DataFrame(rows).pivot(index="period", columns="labels", values="p").round(4))

Step 7. Save the final labels

In [ ]:
out = labels[["ticker", "date", "target_date", "headline", "input_text", "Url",
              "llm_label", "p_bear", "p_neu", "p_bull", "llm_confidence",
              "llm_label_raw", "p_bear_raw", "p_neu_raw", "p_bull_raw"]].copy()
out["llm_model"] = MODEL_NAME
out["prompt_version"] = PROMPT_VERSION
out["text_mode"] = TEXT_MODE
out["calib_T"] = T_CAL
out["calib_w"] = W_CAL

out_path = os.path.join(DATA_DIR, "llm_labels_fnspid.csv")
out.to_csv(out_path, index=False)
print(f"Saved {len(out)} labeled headlines to {out_path}")
print(out["llm_label"].value_counts())

# Twitter check (human vs LLM), for the report
tw_out = tw[["text", "true", "llm", "llm_conf"]].copy()
tw_out[["p_bear_cal", "p_neu_cal", "p_bull_cal"]] = calibrate(p_tw, T_CAL, W_CAL)
tw_out["llm_cal"] = np.array(LABELS)[tw_out[["p_bear_cal", "p_neu_cal", "p_bull_cal"]].to_numpy().argmax(1)]
tw_out.to_csv(os.path.join(DATA_DIR, "llm_check_twitter.csv"), index=False)

# Summary of the price check
result.to_csv(os.path.join(DATA_DIR, "llm_price_check.csv"))

print("\nFiles in DataSource now:")
print(sorted(os.listdir(DATA_DIR)))
drive.flush_and_unmount()

## Key results

| Item | Value |
|---|---|
| LLM | Qwen2.5-7B-Instruct, 4-bit, prompt v1, input = headline + first 300 characters of cleaned body |
| Headlines labeled | 8,784 (after removing repeated headlines) |
| Twitter check, raw LLM (human labels, n = 2,388) | accuracy 0.762, macro-F1 0.736 (always-neutral 0.656; keyword 0.621 / 0.504) |
| Calibration | temperature T = 4, neutral weight w = 4, tuned on half of Twitter, tested on the other half |
| Twitter half B, calibrated (T = 4, w = 4) | accuracy …, macro-F1 … |
| Label mix (calibrated) | 65% neutral, 18% bullish, 17% bearish |
| Price check: bullish − bearish same-day return | LLM calibrated 8.4 pp (t = 15.5) vs keyword 3.1 pp (t = 7.3) |
| Price check in 2021 | LLM calibrated 15.9 pp vs keyword 4.4 pp |

**Main weaknesses:** the raw LLM labels too much neutral news as bullish/bearish (fixed mostly by calibration); a few calibration changes are wrong (e.g. "shares up 7.2%" → neutral); some off-topic articles pass the relevance filter, but the ticker-specific prompt labels them neutral.

In [ ]:
report(calibrate(P[idx_b], T_CAL, W_CAL), f"Half B, final settings (T={T_CAL}, w={W_CAL})")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

tw_out = pd.read_csv(os.path.join(DATA_DIR, "llm_check_twitter.csv"))

# Same split as Step 4b: same seed and same stratification give exactly the same halves
idx_a, idx_b = train_test_split(np.arange(len(tw_out)), test_size=0.5,
                                stratify=tw_out["true"], random_state=SEED)
b = tw_out.iloc[idx_b]

for name, col in [("raw", "llm"), ("calibrated (T=4, w=4)", "llm_cal")]:
    print(f"Half B, {name}: accuracy {accuracy_score(b['true'], b[col]):.3f} | "
          f"macro-F1 {f1_score(b['true'], b[col], average='macro'):.3f}")

print("\nPredicted label mix (calibrated):", b["llm_cal"].value_counts(normalize=True).round(2).to_dict())
print("True label mix:                  ", b["true"].value_counts(normalize=True).round(2).to_dict())